[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/g2_rolling/g2_rolling.ipynb)

# Octonionic Rolling

This example plans the rolling of a marked ball over a fixed sphere three times its radius, rolling without slipping or twisting. A configuration is where the ball touches the sphere and how it is turned, and the question is which rolls lead from one configuration to another. At this ratio of radii a product of configurations, the split-octonion product, reads off rolling: two configurations one roll apart multiply to zero.

A configuration is modelled as a multivector of space without scalar part: its even part is the half turn in the plane square to the contact direction, and its odd part the attitude followed by the reflection in that plane. The product `multiply` multiplies even parts by the geometric product and lets odd parts enter in swapped order. Multiplication by the start, `multiply(start, State)`, is a map whose kernel holds every configuration one roll away, so a route through waypoints comes from a least-squares projection and one more product, not from a search. Maps that preserve the product, built from left and right multiplications, carry routes to routes: they form the exceptional group split G₂.

In the split-octonion notation of Baez and Huerta a contact direction is an imaginary quaternion and an attitude a unit quaternion, the configurations form the projectivized null cone of the imaginary split octonions $\mathrm{Im}\,\mathbb{O}'$, two configurations are one roll apart when $xy = 0$, and the symmetries form $G_2' = \mathrm{Aut}(\mathbb{O}')$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import show_animation
from examples.math.g2_rolling import render

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                          # Scalar+
# The sense of turning that identifies a direction with the plane square to it, as a plane to turn in.
Handedness = ga.gatype.antipseudoscalar()                       # Antipseudoscalar-
Plane = ga.gatype.vector()                                           # Vector+
# Contact points are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()                                       # Antivector-
Rotor = ga.gatype.rotor()                                            # Rotor+
Octonion = ga.gatype.full()                                          # Full+
State = ga.gatype.nonscalar()                                        # Nonscalar+
StateMap = ga.gatype((State, State))
handedness = mv(Handedness, [1])                                     # [] Scalar-

## 1. Configurations and their product

A configuration is modelled as one multivector of space: its even part is the half turn in the plane through the centre square to the contact direction, `handedness * contact`, and its odd part the attitude followed by the reflection in that plane, `-contact.dual() * attitude`. The product couples the two parts. A configuration's two parts have equal size, so its split norm, `pairing(state, state)`, the even part's squared size less the odd part's, vanishes. A configuration is defined only up to scale: its negative, with the antipodal contact, is the same configuration, and choosing the representative whose contact lies on one side of a plane, `lift`, gives the continuous route on the sphere drawn here. At the radius ratio of one to three, each roll along a great circle is a straight line of these states.

The product needs the contact as an element of the attitude's own, extrinsic kind, a half turn, while the contact direction is intrinsic, like every place and direction. In space, and only there, the plane square to a direction is a plane to turn in, of the direction's own grade, and identifying the two takes a sense of turning: `handedness`, an intrinsic unit scalar. The other sense negates the odd part of every configuration, which preserves the product, so which configurations are one roll apart does not depend on it. The handedness may be a cost of the orientation paradigm, or a fact of the subject: the product is built from quaternions, and so belongs to space alone.

In [ ]:
def multiply(first: Octonion, second: Octonion) -> Octonion:
    """The split-octonion product of two multivectors of space, doubling the even part: the even parts multiply
    by the geometric product, and the odd parts enter in swapped order, with reversal for conjugation."""
    first_even, first_odd = first.restrict.even(), first.restrict.odd()        # Even+, Odd+
    second_even, second_odd = second.restrict.even(), second.restrict.odd()    # Even+, Odd+
    # The odd parts enter in the opposite order; reversal supplies their conjugation.
    return (first_even * second_even + second_odd * first_odd.reverse()
            + first_even.reverse() * second_odd + second_even * first_odd)

# The split norm's bilinear form: contact part minus attitude part.
pairing = Octonion.clifford_conjugate().scalar_product(Octonion)     # [] Scalar+ <- (Octonion+, Octonion+)

def encode(contact: Point, attitude: Rotor) -> State:
    """A unit contact point and ball attitude as one null configuration."""
    # The half turn in the plane square to the contact, a bivector of the attitude's own, extrinsic kind, so that
    # the two multiply as quaternions; and the reflection in that plane, through the centre.
    half_turn = handedness * contact                                 # Bivector+
    mirror = contact.dual()                                          # Plane+
    return half_turn - mirror * attitude

def normalize(state: State) -> State:
    """The representative of a configuration whose contact direction has unit length."""
    return state / state.restrict[2].norm()

def decode(state: State) -> tuple[Point, Rotor]:
    """The contact point and ball attitude of a configuration."""
    state = normalize(state)                                         # State+
    contact = state.restrict[2] / handedness                         # Point-
    mirror = contact.dual()                                          # Plane+
    return contact, -mirror.inverse() * state.restrict.odd()

def lift(state: State, hemisphere: Plane) -> State:
    """The representative whose contact lies on the positive side of the plane `hemisphere`, through the
    sphere's centre."""
    contact = state.restrict[2] / handedness                         # Point-
    return normalize(state * (hemisphere & contact))

def roll(start: State, finish: State, fractions: np.ndarray) -> State:
    """The configurations along the projective line between two configurations whose product vanishes: one
    roll."""
    return normalize(start * (1 - fractions) + finish * fractions)

## 2. The waypoint from a product

Two configurations whose product vanishes are connected by one roll along a great circle. If their pairing vanishes, `pairing(start, finish) == 0`, but their product does not, the product `multiply(start, finish)` is the one configuration between them on a route of two rolls.

In [ ]:
attitude_angle = -2.4
sphere_radius = 3
ball_radius = 1
n_samples = 120
n_steps = 41
n_hold_frames = 13
duration_ms = 40
elevation = 35
azimuth = -25
size = 520
samples = np.linspace(0, 1, n_samples)
fractions = np.linspace(0, 1, n_steps)

start_contact = mv.z.dual()                                            # Point-
start_attitude = mv.rotor()                                            # Rotor+
target_contact = mv.x.dual()                                           # Point-
target_attitude = (mv.xy * (-attitude_angle / 2)).exp()                # Rotor+
# The plane through the sphere's centre on whose positive side the drawn contacts lie.
preferred_side = mv.x                                                  # Plane+

In [ ]:
start = encode(start_contact, start_attitude)                          # State+
finish = encode(target_contact, target_attitude)                       # State+
# These endpoints have zero split pairing. Their product supplies the waypoint.
waypoint = lift(multiply(start, finish).cast(State), preferred_side)   # State+
stops = stack([start, waypoint, finish])                               # [stops] State+

def paths(stops: State, fractions: np.ndarray) -> tuple[Point, ...]:
    """The contact points along each leg of a route, sampled for drawing it."""
    states = roll(stops[:-1, None], stops[1:, None], fractions)        # [legs, samples] State+
    contacts, _ = decode(states)                                       # [legs, samples] Point-, Rotor+
    return tuple(contacts)

def rolling(stops: State, fractions: np.ndarray, n_hold_frames: int) -> Iterator[tuple[Point, Rotor]]:
    """The contact and attitude of each frame along the rolls of a route, with its ends held."""
    for _ in range(n_hold_frames):
        yield decode(stops[0])
    for first, second in zip(stops[:-1], stops[1:]):
        for fraction in fractions:
            yield decode(first * (1 - fraction) + second * fraction)
    for _ in range(n_hold_frames):
        yield decode(stops[-1])

show_animation(render.frames(paths(stops, samples), rolling(stops, fractions, n_hold_frames),
                             sphere_radius, ball_radius, elevation, azimuth, size),
               "g2_two_rolls", duration_ms)

The blue and orange arcs are the two rolls. The paint stays fixed to the small ball, making its attitude visible. Each arc is a line in the configuration geometry: its decoded motion rolls without slipping or twisting.

## 3. The route from a kernel

Returning to the same contact point with a different attitude requires a longer route. The kernel of multiplication by the start contains its admissible first waypoints. Requiring zero pairing with the finish leaves configurations that can reach it in two more rolls.

In [ ]:
target_contact = start_contact                                         # Point-
# The half turn the first waypoint is drawn toward: that of a contact square to the preferred side.
preference = handedness * preferred_side.dual()                        # Bivector+

In [ ]:
finish = encode(target_contact, target_attitude)                       # State+
# The least-squares round trip keeps the part detected by multiplication.
# Its complement is the kernel, containing the admissible first rolls.
action = multiply(start, State)                                        # Octonion+ <- State+
reachable = State - action.lstsq(action)                               # State+ <- State+

# Remove the goal-pairing component of the preference, then apply reachability.
remaining = pairing(finish, reachable)                                 # Scalar+ <- State+
first = reachable(preference - remaining.lstsq(remaining(preference)))  # State+
second = multiply(first, finish).cast(State)                           # State+
first = lift(first, preferred_side)                                    # State+
second = lift(second, preferred_side)                                  # State+
stops = stack([start, first, second, finish])                          # [stops] State+

show_animation(render.frames(paths(stops, samples), rolling(stops, fractions, n_hold_frames),
                             sphere_radius, ball_radius, elevation, azimuth, size),
               "g2_three_rolls", duration_ms)

The ball follows the blue, orange and purple arcs and returns to its starting contact with the requested attitude. The route comes from the multiplication kernel and one further product; no search over rolling trajectories is needed.

## 4. A symmetry of whole routes

The invertible maps preserving split-octonion multiplication form split G₂. Such a map preserves every vanishing neighboring product, so it carries an admissible route to another one. Left and right multiplication actions construct an infinitesimal symmetry; the choice below fixes the start while changing the rest of the route.

In [ ]:
symmetry_amount = 0.12
panel_size = 370

In [ ]:
def derivation(first: Octonion, second: Octonion) -> StateMap:
    """An infinitesimal product-preserving map, assembled from multiplication actions."""
    left_first = multiply(first, Octonion)                     # Octonion+ <- Octonion+
    left_second = multiply(second, Octonion)                   # Octonion+ <- Octonion+
    right_first = multiply(Octonion, first)                    # Octonion+ <- Octonion+
    right_second = multiply(Octonion, second)                  # Octonion+ <- Octonion+
    action = (left_first(left_second) - left_second(left_first)
              + left_first(right_second) - right_second(left_first)
              + right_first(right_second) - right_second(right_first))
    return action(State).cast(State)

# This generator's third composition vanishes, so its exponential terminates here.
generator = derivation(start, mv.x)                                    # State+ <- State+
symmetry = State + symmetry_amount * generator + symmetry_amount**2 / 2 * generator(generator)  # State+ <- State+
transformed_stops = symmetry(stops)                                    # [stops] State+

show_animation(render.comparison_frames(
    paths(stops, samples), paths(transformed_stops, samples),
    rolling(stops, fractions, n_hold_frames), rolling(transformed_stops, fractions, n_hold_frames),
    sphere_radius, ball_radius, elevation, azimuth, panel_size),
    "g2_route_symmetry", duration_ms)

The left route and its image on the right begin at the same configuration. The symmetry changes the route and its destination and preserves rolling; it acts on configurations rather than rigidly turning the two spheres.

The construction follows Baez and Huerta's [*G₂ and the Rolling Ball*](https://arxiv.org/html/1205.2447v4), particularly Proposition 7 and Theorem 10. Their global geometry uses projective contact directions and spinor attitudes; each displayed route is a continuous lift to physical rolling on the sphere.

In [ ]:
# checks
# Neighbouring stops of the route, and of its image under the symmetry, multiply to zero: each pair is one
# roll apart.
np.testing.assert_allclose(multiply(stops[:-1], stops[1:]).kernel, 0, atol=1e-10)
np.testing.assert_allclose(multiply(transformed_stops[:-1], transformed_stops[1:]).kernel, 0, atol=1e-10)